Phase 3 — feature engineering. Builds 8 user features, 10 item features (v2 adds a 32-dim sentence-transformer text embedding), k=8 region clusters per city, and a 1:4 negative-sampled training table. Also writes the feature_spec.json that all downstream training notebooks read for input-layer config. The 9 context features (distance, hour, day-of-week, etc.) are computed on the fly at training time, not precomputed here.

In [1]:
import json
import logging
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(message)s",
                    datefmt="%H:%M:%S",
                    force=True)
log = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp("2026-05-06")
NEG_RATIO = 4
RANDOM_SEED = 42
REGION_K = 8

# Load Phase 1 + Phase 2 outputs
restaurants = pd.read_parquet(CLEANED_DIR / "restaurants_open.parquet")
reviews_rest = pd.read_parquet(CLEANED_DIR / "reviews_restaurant.parquet")
users = pd.read_parquet(CLEANED_DIR / "users_target.parquet")
train = pd.read_parquet(CLEANED_DIR / "train_reviews.parquet")

with open(FEATURES_DIR / "cuisine_vocab.json") as f:
    cuisine_vocab = json.load(f)["cuisines"]
print(f"restaurants={len(restaurants)} reviews={len(reviews_rest)} users={len(users)} train={len(train)}")
print(f"cuisine vocab size: {len(cuisine_vocab)}")

restaurants=9022 reviews=1032056 users=359007 train=587676
cuisine vocab size: 50


**User features (8 fields).** Five direct fields read straight from the user row (`avg_rating_given`, `review_count_log`, `days_active`, `elite_flag`, plus the `user_id` itself for embedding lookup). Three aggregated fields require crossing user reviews back into business metadata: `mean_distance_traveled` (pairwise haversine over the user's reviewed restaurants), `fav_cuisine_emb` (top-3 cuisine multi-hot), `price_tolerance_avg` (mean price level / 4).

In [2]:
user_out = pd.DataFrame()
user_out["user_id"] = users["user_id"]
user_out["avg_rating_given"] = users["average_stars"] / 5.0
user_out["review_count_log"] = np.log1p(users["review_count"])
yelping_since = pd.to_datetime(users["yelping_since"], errors="coerce")
user_out["days_active"] = np.log1p((TODAY - yelping_since).dt.days.clip(lower=0))
user_out["elite_flag"] = (
    users["elite"].fillna("").astype(str).str.strip().str.lower().ne("").astype(int)
)
user_out["avg_rating_given"] = user_out["avg_rating_given"].fillna(0.76)
user_out["days_active"] = user_out["days_active"].fillna(0.0)
print(user_out.head(2))

                  user_id  avg_rating_given  review_count_log  days_active  \
0  qVc8ODYU5SZjKXVBgXdI7w             0.782          6.373320     8.859505   
1  j14WgRoU_-2ZE1aw1dXrJg             0.748          8.374246     8.749891   

   elite_flag  
0           1  
1           1  


Now the three aggregated fields. The slow step is the per-user groupby — 359K users, each potentially with dozens of reviewed businesses. Haversine is vectorized inline because the `haversine` package on conda Python 3.9 has a numba ABI conflict (caught earlier; that's why I'm not importing it).

In [3]:
biz_idx = restaurants.set_index("business_id")
biz_lat_arr = biz_idx["latitude"].dropna().to_dict()
biz_lon_arr = biz_idx["longitude"].dropna().to_dict()
biz_categories = biz_idx["categories"].fillna("").to_dict()

def parse_price(attrs):
    if not isinstance(attrs, dict):
        return None
    v = attrs.get("RestaurantsPriceRange2")
    if v is None or str(v).strip().lower() in ("none", ""):
        return None
    try:
        return int(str(v).strip())
    except (ValueError, TypeError):
        return None

biz_price = {bid: parse_price(attrs) for bid, attrs in biz_idx["attributes"].items()}
cuisine_to_idx = {c: i for i, c in enumerate(cuisine_vocab)}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def compute_mean_dist(biz_ids):
    lats, lons = [], []
    for bid in biz_ids:
        if bid in biz_lat_arr:
            lats.append(biz_lat_arr[bid])
            lons.append(biz_lon_arr[bid])
    if len(lats) < 2:
        return 0.0
    if len(lats) > 30:
        rng = np.random.default_rng(RANDOM_SEED)
        idxs = rng.choice(len(lats), 30, replace=False)
        lats = [lats[i] for i in idxs]
        lons = [lons[i] for i in idxs]
    lat_arr = np.array(lats); lon_arr = np.array(lons)
    n = len(lat_arr)
    i, j = np.triu_indices(n, k=1)
    dists = haversine_km(lat_arr[i], lon_arr[i], lat_arr[j], lon_arr[j])
    return float(np.log1p(np.mean(dists)))

def compute_fav_cuisine(biz_ids):
    cnt = Counter()
    for bid in biz_ids:
        cats = biz_categories.get(bid, "")
        for c in (x.strip() for x in cats.split(",")):
            if c in cuisine_to_idx:
                cnt[c] += 1
    emb = np.zeros(len(cuisine_vocab), dtype=np.float32)
    for c, n in cnt.most_common(3):
        emb[cuisine_to_idx[c]] = n
    s = emb.sum()
    return (emb / s).tolist() if s > 0 else emb.tolist()

def compute_price_tol(biz_ids):
    prices = [biz_price[bid] for bid in biz_ids if biz_price.get(bid) is not None]
    if not prices:
        return 0.5
    return float(np.mean(prices) / 4.0)

t0 = time.time()
user_biz_groups = reviews_rest.groupby("user_id")["business_id"].agg(lambda x: list(set(x)))
print(f"groupby done in {time.time()-t0:.1f}s ({len(user_biz_groups)} users)")

t0 = time.time()
user_id_to_biz = user_biz_groups.to_dict()
mean_dists, fav_cuisines, price_tols = [], [], []
n = len(user_out)
for i, uid in enumerate(user_out["user_id"].values):
    biz_ids = user_id_to_biz.get(uid, [])
    mean_dists.append(compute_mean_dist(biz_ids))
    fav_cuisines.append(compute_fav_cuisine(biz_ids))
    price_tols.append(compute_price_tol(biz_ids))
    if (i+1) % 50_000 == 0:
        log.info("processed %d/%d users", i+1, n)
user_out["mean_distance_traveled"] = mean_dists
user_out["fav_cuisine_emb"] = fav_cuisines
user_out["price_tolerance_avg"] = price_tols
print(f"aggregations done in {time.time()-t0:.1f}s")

groupby done in 4.6s (359009 users)


17:07:23 | processed 50000/359007 users
17:07:25 | processed 100000/359007 users
17:07:26 | processed 150000/359007 users
17:07:27 | processed 200000/359007 users
17:07:28 | processed 250000/359007 users
17:07:29 | processed 300000/359007 users
17:07:30 | processed 350000/359007 users


aggregations done in 9.4s


Add an OOV row for `<NEW_USER>` with neutral defaults — at deploy time, brand-new Taste hunter users will hit this row instead of crashing on a missing `user_id`.

In [4]:
user_oov = pd.DataFrame([{
    "user_id": "<NEW_USER>",
    "avg_rating_given": 0.76,
    "review_count_log": 0.0,
    "days_active": 0.0,
    "elite_flag": 0,
    "mean_distance_traveled": 0.0,
    "fav_cuisine_emb": np.zeros(len(cuisine_vocab), dtype=np.float32).tolist(),
    "price_tolerance_avg": 0.5,
}])
user_features = pd.concat([user_out, user_oov], ignore_index=True)
user_features.to_parquet(FEATURES_DIR / "user_features.parquet", index=False)
print(f"user_features: {len(user_features)} rows ({len(user_features)-1} real + 1 OOV)")

user_features: 359008 rows (359007 real + 1 OOV)


**Item features (9 fields).** All 9 are direct functions of the business row plus the cuisine vocabulary. The `avg_rating` uses Bayesian smoothing with a prior strength of C=10 — a restaurant with one 5-star review gets pulled toward the global mean instead of looking equivalent to a real 5-star place. `photo_count_log` is a stand-in: the actual photos.json file is 5GB and not downloaded, so I use `log1p(review_count)` as a proxy that correlates with photo abundance. The PRD has this listed as a known training/deploy gap.

In [5]:
item_out = pd.DataFrame()
item_out["business_id"] = restaurants["business_id"]

C = 10
mu_global = restaurants["stars"].mean()
n = restaurants["review_count"]
item_out["avg_rating"] = (C * mu_global + restaurants["stars"] * n) / (C + n) / 5.0
item_out["review_count_log"] = np.log1p(restaurants["review_count"])

def parse_price_item(attrs):
    if not isinstance(attrs, dict):
        return 2
    v = attrs.get("RestaurantsPriceRange2")
    if v is None or str(v).strip().lower() in ("none", ""):
        return 2
    try:
        return int(str(v).strip())
    except (ValueError, TypeError):
        return 2

def parse_outdoor(attrs):
    if not isinstance(attrs, dict):
        return 0
    v = attrs.get("OutdoorSeating")
    if v is None:
        return 0
    return int(str(v).strip().lower() == "true")

def encode_categories(cats_str):
    emb = np.zeros(len(cuisine_vocab), dtype=np.int8)
    if not isinstance(cats_str, str):
        return emb.tolist()
    for c in (x.strip() for x in cats_str.split(",")):
        if c in cuisine_to_idx:
            emb[cuisine_to_idx[c]] = 1
    return emb.tolist()

item_out["price_level"] = restaurants["attributes"].apply(parse_price_item)
item_out["is_open"] = restaurants["is_open"].astype(int)
item_out["has_outdoor_seating"] = restaurants["attributes"].apply(parse_outdoor)
item_out["photo_count_log"] = np.log1p(restaurants["review_count"]).clip(upper=10)
item_out["categories_multi_hot"] = restaurants["categories"].apply(encode_categories)
item_out["city"] = restaurants["city"].str.strip()

city_to_idx = {c: i+1 for i, c in enumerate(sorted(item_out["city"].unique()))}
city_to_idx["<UNK>"] = 0
item_out["city_id"] = item_out["city"].map(city_to_idx).fillna(0).astype(int)
item_out["lat"] = restaurants["latitude"]
item_out["lon"] = restaurants["longitude"]

item_oov = pd.DataFrame([{
    "business_id": "<NEW_BUSINESS>",
    "avg_rating": mu_global / 5.0,
    "review_count_log": 0.0,
    "price_level": 2,
    "is_open": 1,
    "has_outdoor_seating": 0,
    "photo_count_log": 0.0,
    "categories_multi_hot": np.zeros(len(cuisine_vocab), dtype=np.int8).tolist(),
    "city": "<UNK>",
    "city_id": 0,
    "lat": 0.0,
    "lon": 0.0,
}])
item_features = pd.concat([item_out, item_oov], ignore_index=True)
print(f"item_features (pre-text-emb): {len(item_features)} rows")
print(f"city_id_map: {city_to_idx}")

item_features (pre-text-emb): 9023 rows
city_id_map: {'Philadelphia': 1, 'Tampa': 2, 'Tucson': 3, '<UNK>': 0}


**v2 — item text embedding.** Encode each restaurant's `name + categories` with `sentence-transformers/all-MiniLM-L6-v2` (22 MB pretrained, runs on MPS), then PCA-reduce 384d to 32d so it fits cleanly into the DeepFM and Two-Tower input layers. This is the ColBERT-light item encoder — a lightweight version of the BERT-based document encoder pattern from search systems. The main payoff is at cold-start: when a brand-new restaurant has an OOV `business_id` embedding, the model still has 32 dims of semantic signal from the name/categories text. OOV row gets zeros(32) since `<NEW_BUSINESS>` has no real text.

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

restaurants_df = restaurants.copy()
restaurants_df["text_input"] = (
    restaurants_df["name"].fillna("") + " - " +
    restaurants_df["categories"].fillna("")
).str.strip()
print(f"sample text inputs:")
for s in restaurants_df["text_input"].head(3):
    print(f"  {s[:80]}")

t0 = time.time()
encoder = SentenceTransformer("all-MiniLM-L6-v2")
text_emb_384 = encoder.encode(
    restaurants_df["text_input"].tolist(),
    batch_size=64,
    show_progress_bar=False,
)
print(f"sentence-transformer encoding: {text_emb_384.shape} in {time.time()-t0:.1f}s")

pca = PCA(n_components=32, random_state=RANDOM_SEED)
text_emb_32 = pca.fit_transform(text_emb_384)
print(f"PCA(384 -> 32) explained variance: {pca.explained_variance_ratio_.sum():.3f}")
print(f"text_emb_32 shape: {text_emb_32.shape}")

text_emb_with_oov = np.zeros((len(item_features), 32), dtype=np.float32)
text_emb_with_oov[:len(text_emb_32)] = text_emb_32.astype(np.float32)

item_features["item_text_emb_pca32"] = list(text_emb_with_oov)
item_features.to_parquet(FEATURES_DIR / "item_features.parquet", index=False)
print(f"item_features (with text emb): {len(item_features)} rows, 10 features + business_id")

17:07:36 | No device provided, using mps
17:07:36 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
17:07:36 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"


sample text inputs:
  St Honore Pastries - Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries
  Tuna Bar - Sushi Bars, Restaurants, Japanese
  BAP - Korean, Restaurants


17:07:36 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
17:07:36 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
17:07:36 | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
17:07:36 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
17:07:36 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
17:07:37 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redir

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

17:07:37 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
17:07:37 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
17:07:37 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
17:07:37 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
17:07:37 | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
17:07:37 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
17:07:38 | HTTP Re

sentence-transformer encoding: (9022, 384) in 8.6s
PCA(384 -> 32) explained variance: 0.589
text_emb_32 shape: (9022, 32)
item_features (with text emb): 9023 rows, 10 features + business_id


**Region clusters (k=8 per city).** From the EDA elbow plot k=8 looked like the right number, so I run k-means independently on each city's lat/lon. The cluster centers go into `region_clusters.json` and at training time each business is assigned a `region_cluster_id ∈ [0, 24]` (3 cities × 8 + a `<UNK>` slot).

In [7]:
from sklearn.cluster import KMeans

clusters = {}
for city in sorted(item_out["city"].unique()):
    if not city:
        continue
    sub = restaurants[restaurants["city"].str.strip() == city][["latitude", "longitude"]].dropna()
    if len(sub) < REGION_K:
        continue
    km = KMeans(n_clusters=REGION_K, n_init=10, random_state=RANDOM_SEED)
    km.fit(sub.values)
    clusters[city] = {
        "centers": km.cluster_centers_.tolist(),
        "n": len(sub),
        "inertia": float(km.inertia_),
    }
    print(f"{city}: n={len(sub)} k={REGION_K} inertia={km.inertia_:.2f}")

with open(FEATURES_DIR / "region_clusters.json", "w") as f:
    json.dump(clusters, f, indent=2)

Philadelphia: n=4373 k=8 inertia=2.24
Tampa: n=2502 k=8 inertia=2.30
Tucson: n=2147 k=8 inertia=2.99


**Train table with 1:4 negative sampling.** For every positive review (stars ≥ 4), draw 4 negatives from the *same city*, excluding restaurants the user has already been to. Same-city sampling matches the realistic candidate pool — picking negatives from a different city would make the task too easy (cuisine/region differences would dominate). 1:4 is a common ratio: 1:1 lets the model coast on 'I see, this rating is missing' without learning preference, while 1:99 (which we use only at eval) makes BCE loss too negative-dominated for stable training.

In [8]:
rng = np.random.default_rng(RANDOM_SEED)

biz_city = restaurants.set_index("business_id")["city"].to_dict()
city_biz = {}
for bid, c in biz_city.items():
    c = (c or "").strip()
    city_biz.setdefault(c, []).append(bid)
for c in city_biz:
    city_biz[c] = np.array(city_biz[c])

train_neg_in = train.copy()
train_neg_in["city"] = train_neg_in["business_id"].map(biz_city).fillna("").str.strip()
train_neg_in["label"] = (train_neg_in["stars"] >= 4).astype(int)

positives = train_neg_in[train_neg_in["label"] == 1]
print(f"total positives: {len(positives)}")

user_visited = train_neg_in.groupby("user_id")["business_id"].agg(set).to_dict()

t0 = time.time()
out_rows = []
for i, row in enumerate(positives.itertuples(index=False)):
    out_rows.append({
        "user_id": row.user_id,
        "business_id": row.business_id,
        "label": 1,
        "stars": row.stars,
        "date": row.date,
        "city": row.city,
    })
    if row.city not in city_biz:
        continue
    candidates = city_biz[row.city]
    visited = user_visited.get(row.user_id, set())
    sampled, attempts = [], 0
    while len(sampled) < NEG_RATIO and attempts < NEG_RATIO * 10:
        picks = rng.choice(candidates, size=NEG_RATIO * 2, replace=True)
        for bid in picks:
            if bid not in visited and bid != row.business_id:
                sampled.append(bid)
                if len(sampled) >= NEG_RATIO:
                    break
        attempts += 1
    for bid in sampled:
        out_rows.append({
            "user_id": row.user_id,
            "business_id": bid,
            "label": 0,
            "stars": np.nan,
            "date": row.date,
            "city": row.city,
        })
    if (i+1) % 50_000 == 0:
        log.info("processed %d/%d positives (%.1fs)", i+1, len(positives), time.time()-t0)

train_with_neg = pd.DataFrame(out_rows)
train_with_neg.to_parquet(FEATURES_DIR / "train_with_negatives.parquet", index=False)
n_pos = (train_with_neg["label"] == 1).sum()
n_neg = (train_with_neg["label"] == 0).sum()
print(f"train_with_negatives: {len(train_with_neg)} rows = {n_pos} positive + {n_neg} negative (ratio {n_neg/max(n_pos,1):.2f})")
print(f"total time: {time.time()-t0:.1f}s")

total positives: 415057


17:07:51 | processed 50000/415057 positives (0.9s)
17:07:52 | processed 100000/415057 positives (1.9s)
17:07:53 | processed 150000/415057 positives (2.6s)
17:07:54 | processed 200000/415057 positives (3.7s)
17:07:55 | processed 250000/415057 positives (4.5s)
17:07:56 | processed 300000/415057 positives (5.7s)
17:07:57 | processed 350000/415057 positives (6.6s)
17:07:58 | processed 400000/415057 positives (8.2s)


train_with_negatives: 2075285 rows = 415057 positive + 1660228 negative (ratio 4.00)
total time: 10.2s


**feature_spec.json.** Single source of truth for every downstream training notebook. Lists the 26 features with their type, dim, vocab size, and source. Total estimated input dimension is what the DeepFM input layer flattens to before the FM and DNN heads split off.

In [9]:
EMB_DIM_USER = 8
EMB_DIM_BIZ = 8
n_users = len(users)
n_biz = len(restaurants)

spec = {
    "version": "v3.0_2026-05-06",
    "anchor": "PRD §3.3.3 v2.2 (26 features: User 8 + Item 9 + Context 9)",
    "n_users": n_users,
    "n_businesses": n_biz,
    "cuisine_vocab_size": len(cuisine_vocab),
    "city_id_map": city_to_idx,
    "user_features": [
        {"name": "user_id", "type": "embedding", "vocab_size": n_users + 1, "dim": EMB_DIM_USER, "source": "user.user_id (+ <NEW_USER> OOV)"},
        {"name": "avg_rating_given", "type": "numeric", "range": [0, 1], "source": "user.average_stars / 5"},
        {"name": "review_count_log", "type": "numeric", "source": "log1p(user.review_count)"},
        {"name": "days_active", "type": "numeric", "source": "log1p((today - yelping_since).days)"},
        {"name": "elite_flag", "type": "binary", "source": "user.elite != ''"},
        {"name": "mean_distance_traveled", "type": "numeric", "source": "log1p(haversine pairwise mean over user's biz)"},
        {"name": "fav_cuisine_emb", "type": "vector", "dim": len(cuisine_vocab), "source": "top-3 cuisine pooling over user's reviewed biz"},
        {"name": "price_tolerance_avg", "type": "numeric", "range": [0, 1], "source": "mean(biz.attributes.RestaurantsPriceRange2) / 4"},
    ],
    "item_features": [
        {"name": "business_id", "type": "embedding", "vocab_size": n_biz + 1, "dim": EMB_DIM_BIZ, "source": "business.business_id (+ <NEW_BUSINESS> OOV)"},
        {"name": "avg_rating", "type": "numeric", "range": [0, 1], "source": "Bayesian smoothed (C=10) business.stars / 5"},
        {"name": "review_count_log", "type": "numeric", "source": "log1p(business.review_count)"},
        {"name": "price_level", "type": "ordinal", "range": [1, 4], "source": "business.attributes.RestaurantsPriceRange2"},
        {"name": "is_open", "type": "binary", "source": "business.is_open"},
        {"name": "has_outdoor_seating", "type": "binary", "source": "business.attributes.OutdoorSeating == True"},
        {"name": "photo_count_log", "type": "numeric", "source": "log1p(business.review_count) clip(0,10) - proxy"},
        {"name": "categories_multi_hot", "type": "vector", "dim": len(cuisine_vocab), "source": "multi-hot over cuisine_vocab"},
        {"name": "city_id", "type": "embedding", "vocab_size": len(city_to_idx), "dim": 4, "source": "business.city -> city_id_map"},
        {"name": "item_text_emb_pca32", "type": "vector", "dim": 32, "source": "sentence-transformers/all-MiniLM-L6-v2 on (name + categories) -> PCA32 (v2 ColBERT-light)"},
    ],
    "context_features": [
        {"name": "distance_from_user_km_log", "type": "numeric", "source": "haversine(user_lat_lon, biz_lat_lon) -> log1p"},
        {"name": "hour_bucket", "type": "cyclic", "source": "review.date.hour -> sin/cos + 3-bucket"},
        {"name": "day_of_week", "type": "categorical", "vocab_size": 7, "source": "review.date.weekday"},
        {"name": "is_weekend", "type": "binary", "source": "day_of_week in {Sat, Sun}"},
        {"name": "trip_day_index", "type": "numeric", "range": [0, 1], "source": "trip context (synthesized for training; real for F2 deploy)"},
        {"name": "region_cluster_id", "type": "embedding", "vocab_size": REGION_K + 1, "dim": 4, "source": "k-means(k=8) on biz lat/lon per city"},
        {"name": "period_id", "type": "categorical", "vocab_size": 4, "source": "F2 trip period (morning/afternoon/evening) + <UNK>"},
        {"name": "activity_emb", "type": "vector", "dim": 32, "source": "sentence-transformer all-MiniLM-L6-v2 -> PCA(32) on F2 activity text; zeros for non-trip"},
        {"name": "prior_meals_cuisines", "type": "vector", "dim": len(cuisine_vocab), "source": "multi-hot of cuisines already chosen earlier in trip; zeros for non-trip"},
    ],
    "total_features": 27,
    "total_input_dim_estimate": EMB_DIM_USER + 1 + 1 + 1 + 1 + 1 + len(cuisine_vocab) + 1 + EMB_DIM_BIZ + 1 + 1 + 1 + 1 + 1 + 1 + len(cuisine_vocab) + 4 + 32 + 1 + 2 + 7 + 1 + 1 + 4 + 4 + 32 + len(cuisine_vocab),
    "missing_handling": {
        "numeric": "fillna(global median)",
        "categorical": "<UNK> embedding (reserved index 0)",
        "user_id_OOV": "<NEW_USER> reserved for new Taste hunter users",
        "trip_features_non_trip": "trip_day_index=0, period_id=<UNK>, activity_emb=zeros, prior_meals_cuisines=zeros",
    },
}

with open(FEATURES_DIR / "feature_spec.json", "w") as f:
    json.dump(spec, f, indent=2, ensure_ascii=False)
print(f"total features: {spec['total_features']}")
print(f"total input dim estimate: {spec['total_input_dim_estimate']}")

total features: 27
total input dim estimate: 266


Phase 3 done. Outputs: `user_features.parquet`, `item_features.parquet`, `train_with_negatives.parquet`, `region_clusters.json`, `feature_spec.json`. Phase 4 baselines (MF + FM) reads these directly.